# 🚀 02. Huấn luyện Vision-Only Student Model (Giai đoạn 2 - CMKD)

> **Dự án:** Nhận diện quả sầu riêng chín hay không bằng Chưng cất tri thức Đa phương thức (CMKD)  
> **Phụ trách:** Leader / Member 1 (Vision Lead & Multi-Modal Framework)  
> **Tham chiếu:** `specification/Đặc tả đề tài.pdf` (Mục 1.3.3) & `tmp_student.md`  
> **Vai trò của Notebook:**
1. Kết nối Google Drive & kiểm tra GPU.
2. `git clone` / `git pull` mã nguồn mới nhất từ GitHub (nhánh `cocobaut`).
3. Liên kết dữ liệu (`data/final`), trọng số Teacher (`checkpoints/teachermodal/best_teacher.pth`) và tạo thư mục đích (`checkpoints/studentmodal`, `results_student`) trên Google Drive.
4. Kích hoạt huấn luyện chưng cất 2 pha (`train_student.py`) và xuất bảng đối sánh thực nghiệm.

In [1]:
# ==============================================================================
# 1. KẾT NỐI GOOGLE DRIVE & KIỂM TRA GPU
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("✅ Đã kết nối Google Drive thành công!")
else:
    print("ℹ️ Đang chạy trên môi trường Local.")

!nvidia-smi

Mounted at /content/drive
✅ Đã kết nối Google Drive thành công!
Mon Nov  9 14:15:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |        

In [2]:
# ==============================================================================
# 2. ĐỒNG BỘ CODE MỚI NHẤT TỪ GITHUB (NHÁNH cocobaut)
# ==============================================================================
REPO_URL = "https://github.com/Cocobaut/durian-ripeness-classification.git"
BRANCH = "cocobaut"
REPO_DIR = "/content/durian-ripeness-classification"

if IN_COLAB:
    %cd /content
    if not os.path.exists(REPO_DIR):
        !git clone -b $BRANCH $REPO_URL $REPO_DIR
        %cd $REPO_DIR
    else:
        %cd $REPO_DIR
        # Gỡ symlink tạm trong repo (không ảnh hưởng dữ liệu trên Drive) để tránh xung đột khi git pull
        !find . -maxdepth 5 -type l -delete
        !git fetch origin
        !git checkout -f $BRANCH
        !git reset --hard origin/$BRANCH
else:
    cur = os.path.abspath(".")
    while not (os.path.isdir(os.path.join(cur, "src")) and os.path.isdir(os.path.join(cur, "workspaces"))):
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    os.chdir(cur)

print(f"✅ Đang làm việc tại thư mục gốc dự án: {os.getcwd()}")

/content
Cloning into '/content/durian-ripeness-classification'...
remote: Enumerating objects: 533, done.
remote: Counting objects: 100% (533/533), done.
remote: Compressing objects: 100% (395/395), done.
remote: Total 533 (delta 211), reused 439 (delta 127), pack-reused 0 (from 0)
Receiving objects: 100% (533/533), 24.30 MiB | 20.38 MiB/s, done.
Resolving deltas: 100% (211/211), done.
/content/durian-ripeness-classification
✅ Đang làm việc tại thư mục gốc dự án: /content/durian-ripeness-classification


In [3]:
# ==============================================================================
# 3. CÀI ĐẶT THƯ VIỆN PHỤ THUỘC SIÊU TỐC VỚI `uv`
# ==============================================================================
!pip install -q uv
!uv pip install --system -r requirements.txt
print("✅ Đã hoàn tất cài đặt toàn bộ thư viện!")

Using Python 3.13.16 environment at: /usr
Resolved 167 packages in 858ms                                       
Prepared 13 packages in 1.38s                                            
Uninstalled 3 packages in 21ms
Installed 13 packages in 112ms                              
 + async-lru==2.4.0
 + comm==0.2.3
 - ipykernel==6.17.1
 + ipykernel==7.4.0
 + jedi==0.20.1
 + json5==0.17.3
 + jupyter==1.1.1
 + jupyter-builder==1.2.4
 - jupyter-client==7.4.9
 + jupyter-client==8.10.0
 + jupyter-lsp==2.3.1
 + jupyterlab==4.6.6
 + jupyterlab-server==2.28.1
 + nest-asyncio2==1.7.4
 - notebook==6.5.7
 + notebook==7.6.5
✅ Đã hoàn tất cài đặt toàn bộ thư viện!


In [4]:
# ==============================================================================
# 4. THIẾT LẬP ĐƯỜNG DẪN CHECKPOINTS & DATASET TỪ GOOGLE DRIVE
# ==============================================================================
from pathlib import Path

if IN_COLAB:
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/durian-ripeness-cmkd")
    DRIVE_CHECKPOINTS = DRIVE_PROJECT_ROOT / "checkpoints"
    DRIVE_DATA = DRIVE_PROJECT_ROOT / "data" / "final"
    DRIVE_RESULTS_TEACHER = DRIVE_PROJECT_ROOT / "results_teacher"
    DRIVE_RESULTS_STUDENT = DRIVE_PROJECT_ROOT / "results_student"

    # Tạo sẵn folder studentmodal & results_student trên Google Drive để lưu kết quả thời gian thực
    (DRIVE_CHECKPOINTS / "teachermodal").mkdir(parents=True, exist_ok=True)
    (DRIVE_CHECKPOINTS / "studentmodal").mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS_STUDENT.mkdir(parents=True, exist_ok=True)

    # Tạo symlink từ repo tới Google Drive (mất mạng hay sập máy ảo Colab cũng không mất file)
    !rm -rf checkpoints/unimodal checkpoints/teachermodal checkpoints/studentmodal
    !rm -rf workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
    !mkdir -p checkpoints data/final
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'unimodal')}" checkpoints/unimodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'teachermodal')}" checkpoints/teachermodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'studentmodal')}" checkpoints/studentmodal
    !ln -sfn "{str(DRIVE_RESULTS_STUDENT)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
    if DRIVE_RESULTS_TEACHER.exists():
        !rm -rf workspaces/member1_vision_lead/04_train_teacher_and_student/results
        !ln -sfn "{str(DRIVE_RESULTS_TEACHER)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results
    for split_name in ["train", "valid", "test"]:
        src_split = DRIVE_DATA / split_name
        if src_split.exists():
            !rm -rf data/final/{split_name}
            !ln -sfn "{str(src_split)}" data/final/{split_name}

Path("checkpoints/studentmodal").mkdir(parents=True, exist_ok=True)
print("✅ Kiểm tra trạng thái đường dẫn:")
print(" - Teacher ckpt   :", Path("checkpoints/teachermodal/best_teacher.pth").exists(), "->", Path("checkpoints/teachermodal").resolve())
print(" - Studentmodal   :", Path("checkpoints/studentmodal").resolve())
print(" - Results student:", Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results_student").resolve())
print(" - Manifest CSV   :", Path("data/final/final_multimodal_manifest.csv").exists())

✅ Kiểm tra trạng thái đường dẫn:
 - Teacher ckpt   : True -> /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/teachermodal
 - Studentmodal   : /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/studentmodal
 - Results student: /content/drive/MyDrive/durian-ripeness-cmkd/results_student
 - Manifest CSV   : True


In [5]:
# ==============================================================================
# 5. THỰC THI HUẤN LUYỆN CHƯNG CẤT VISION-ONLY STUDENT (GIAI ĐOẠN 2 - CMKD)
# ==============================================================================
!python workspaces/member1_vision_lead/04_train_teacher_and_student/train_student.py \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --teacher-ckpt checkpoints/teachermodal/best_teacher.pth \
    --student-backbone convnextv2_tiny \
    --save-path checkpoints/studentmodal/best_student.pth \
    --results-dir workspaces/member1_vision_lead/04_train_teacher_and_student/results_student \
    --alpha 1.0 \
    --beta 2.0 \
    --gamma 1.0 \
    --temperature 3.0 \
    --feat-loss-type cosine \
    --loss-type cost_sensitive_focal \
    --focal-gamma 2.0 \
    --cost-weight 0.5 \
    --batch-size 16 \
    --epochs 30 \
    --warmup-epochs 3 \
    --lr 5e-4 \
    --lr-backbone 1e-4 \
    --min-lr 1e-6 \
    --patience 10 \
    --select-metric fruit_macro_f1 \
    --eval-test \
    --device cuda

[2026-11-09 14:16:28] ==============================================================================
[2026-11-09 14:16:28] HUẤN LUYỆN VISION-ONLY STUDENT BẰNG CMKD (GIAI ĐOẠN 2)
[2026-11-09 14:16:28] Thiết bị: cuda | AMP: True (torch.bfloat16) | batch 16 x accum 1
[2026-11-09 14:16:28] Student backbone : convnextv2_tiny.fcmae_ft_in1k
[2026-11-09 14:16:28] Distill weights  : alpha=1.0 (CE) | beta=2.0 (KD, T=3.0) | gamma=1.0 (Feat: cosine)
[2026-11-09 14:16:28] Best student ckpt: /content/durian-ripeness-classification/checkpoints/studentmodal/best_student.pth
[2026-11-09 14:16:28] ==============================================================================
[2026-11-09 14:16:28] Nạp Frozen Teacher từ: /content/durian-ripeness-classification/checkpoints/teachermodal/best_teacher.pth
/content/durian-ripeness-classification/src/datasets/sensor_dataset.py:132: UserWarning: Không nạp được /content/durian-ripeness-classification/workspaces/member3_sensor/02_train_unimodal_sensor/results/sens

In [6]:
# ==============================================================================
# 6. ĐÁNH GIÁ ĐỘC LẬP (RGB-ONLY) & HIỂN THỊ BẢNG ĐỐI SÁNH BENCHMARK
# ==============================================================================
!python workspaces/member1_vision_lead/04_train_teacher_and_student/eval_student.py \
    --checkpoint checkpoints/studentmodal/best_student.pth \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --split test \
    --output-dir workspaces/member1_vision_lead/04_train_teacher_and_student/results_student

comp_md = Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results_student/benchmark_comparison.md")
if comp_md.is_file():
    print("\n" + comp_md.read_text(encoding="utf-8"))
print("✅ Toàn bộ kết quả đánh giá Student đã được lưu tại:", comp_md.parent.resolve())

[2026-11-09 15:13:44] [INFO] [eval_teacher]: Khởi tạo TeacherEvaluator trên thiết bị: cuda (input_mode=vision)
[2026-11-09 15:13:44] [INFO] [eval_teacher]: Kết quả đánh giá sẽ được lưu tại: /content/durian-ripeness-classification/workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
[2026-11-09 15:13:44] [INFO] [eval_teacher]: === BẮT ĐẦU QUY TRÌNH ĐÁNH GIÁ TOÀN DIỆN TEACHER MODEL ===
[2026-11-09 15:13:44] [INFO] [eval_teacher]: model=vision_student | split=test | input_mode=vision | strategy=mean_prob
[2026-11-09 15:13:44] [INFO] [eval_teacher]: Bắt đầu đánh giá cấp độ sample (mask_modality=None)...
[2026-11-09 15:13:52] [INFO] [eval_teacher]: Đã suy luận 298 sample, accuracy = 0.4966
[2026-11-09 15:13:52] [INFO] [eval_teacher]: Tổng hợp kết quả cấp độ quả (fruit_id) theo chiến lược: mean_prob...
[2026-11-09 15:13:52] [INFO] [eval_teacher]: Đã tổng hợp 24 quả, accuracy cấp quả = 0.5833
[2026-11-09 15:13:53] [INFO] [eval_teacher]: Bắt đầu phân tích đóng góp phương